In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

import math
import copy
from dataclasses import dataclass
from functools import lru_cache

torch.manual_seed(0)

In [ ]:
@dataclass
class ModelArgs:
    vocab_size : int = 129_280
    dim : int = 4_096
    n_layers : int = 43
    norm_eps : float = 1e-6
    
    n_heads : int = 64
    head_dim : int = 512
    rope_head_dim : int = 64 # attends only last rope_head_dim dims out of head_dim.
    q_lora_rank : int = 1_024 # query latent size d_c
    o_groups : int = 8 # grouped output projection : g
    o_lora_rank : int = 1_024 # d_g
    window_size : int = 128 # sliding window n_win
    compress_ratios : tuple = (0, 0) + (4, 128) * 20 + (4, 0, 0, 0) # 0 = SWA only, 4 = CSA, 128 = HCA, last 3 zeros dspark.
    
    rope_theta: float = 10000.0 # pure-SWA layers, no YaRN
    compress_rope_theta: float = 160000.0 # CSA / HCA layers, with YaRN
    original_seq_len : int = 65_536
    rope_factor : float = 16.0
    beta_fast : int = 32
    beta_slow : int = 1
    max_seq_len : int = 4096 # length of the RoPE precomputed table

    # lightning indexer - CSA
    index_n_heads : int = 64
    index_head_dim : int = 128
    index_topk : int = 512
    
    # MoE
    n_routed_experts : int = 256
    n_shared_experts : int = 1
    n_activated_experts : int = 6
    n_hash_layers : int = 3 # layers 0, 1, 2 pick experts by token id
    moe_inter_dim : int = 2_048
    score_func : str = "sqrtsoftplus" # routing score function - V4 only uses sqrt(softplus)
    route_scale : float = 1.5
    swiglu_limit : float = 10.0
    
    # mHC
    hc_mult: int = 4
    hc_sinkhorn_iters: int = 20
    hc_eps: float = 1e-6

    # dSpark
    n_mtp_layers : int = 3
    dspark_block_size : int = 5 # tokens per guessed block, 0 means no drafter
    dspark_noise_token_id : int = 128_799 # placeholder for the not-yet-known tokens
    dspark_target_layer_ids : tuple = (40, 41, 42) # main layers the drafter reads
    dspark_markov_rank : int = 256 # bi-gram part

    # checkpoint formats
    dtype : str = "fp8"
    scale_fmt : str = "ue8m0"
    expert_dtype : str = "fp4"

    def __post_init__(self):
        self.compress_ratios = tuple(self.compress_ratios) # JSON gives a list
        self.dspark_target_layer_ids = tuple(self.dspark_target_layer_ids)
        assert len(self.compress_ratios) >= self.n_layers + self.n_mtp_layers # HF list: 43 main layers + d DSpark stages
        assert self.rope_head_dim % 2 == 0 and self.rope_head_dim < self.head_dim
        assert self.index_head_dim & (self.index_head_dim - 1) == 0 # Hadamard needs a power of 2
        assert (self.n_heads * self.head_dim) % self.o_groups == 0
        assert self.n_shared_experts == 1
        assert self.score_func == "sqrtsoftplus"

In [ ]:
def tiny_args(dspark : bool = False) -> ModelArgs:
    return ModelArgs(
        vocab_size = 512,
        dim = 128,
        n_layers = 6,
        n_heads = 4,
        head_dim = 80,
        rope_head_dim = 16,
        q_lora_rank = 64,
        o_groups = 2,
        o_lora_rank = 32,
        window_size = 8,
        compress_ratios = (0, 0, 4, 16, 4, 16, 0, 0, 0),
        max_seq_len = 256,
        index_n_heads = 4,
        index_head_dim = 32,
        index_topk = 4,
        n_routed_experts = 8,
        n_activated_experts = 2,
        moe_inter_dim = 64,
        dspark_block_size = 5 if dspark else 0,
        dspark_noise_token_id = 500,
        dspark_target_layer_ids = (3, 4, 5),
        dspark_markov_rank = 16
    )

In [ ]:
def upcast(x : torch.Tensor):
    return x.float() if x.dtype in (torch.float16, torch.bfloat16) else x

class RMSNorm(nn.Module):
    def __init__(self, dim : int, eps : float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    
    def forward(self, x : torch.Tensor):
        dtype = x.dtype
        x = upcast(x)
        x_norm = x * torch.rsqrt(x.square().mean(dim = -1, keepdim = True) + self.eps)
        return (self.weight * x_norm).to(dtype)

In [ ]:
@lru_cache(2) # 2 kinds of cache - SWA, CSA / HCA, once stored, not calculated again and again.
def precompute_theta_cis(
    dim : int,
    seq_len : int,
    original_seq_len : int,
    base : float,
    factor : float,
    beta_fast : int,
    beta_slow : int
    
):
    freqs = 1 / ((base) ** (torch.arange(0, dim, 2, dtype = torch.float32) / dim))
    if original_seq_len > 0: # YaRN
        def pair_index(n_turns):
            return dim * math.log(original_seq_len / (2 * math.pi * n_turns)) / (2 * math.log(base))
        
        low = max(math.floor(pair_index(beta_fast)), 0)
        high = min(math.ceil(pair_index(beta_slow)), dim - 1)

        if low == high:
            high += 0.001
        
        ramp = ((torch.arange(0, dim // 2, dtype = torch.float32) - low) / (high - low)).clamp(0, 1)
        freqs = ((1 - ramp) * (freqs)) + (ramp * (freqs / factor))
    
    angles = torch.outer(torch.arange(seq_len), freqs) # shape [seq_len, dim // 2]
    return torch.polar(torch.ones_like(angles), angles) # e ^ (i * angles)

In [ ]:
def layer_freqs_cis(args : ModelArgs, compress_ratio : int):
    if compress_ratio > 0: # CSA / HCA, YaRN
        return precompute_theta_cis(
            args.rope_head_dim, 
            args.max_seq_len, 
            args.original_seq_len, 
            args.compress_rope_theta, 
            args.rope_factor, 
            args.beta_fast, 
            args.beta_slow
        )
    
    return precompute_theta_cis(
        args.rope_head_dim, 
        args.max_seq_len, 
        0, 
        args.rope_theta, 
        args.rope_factor, 
        args.beta_fast, 
        args.beta_slow
    )

In [ ]:
def apply_rope(x : torch.Tensor, freqs_cis : torch.Tensor, inverse : bool = False) -> torch.Tensor:
    rope_dim = freqs_cis.shape[-1] * 2
    x_pass, x_rot = x[..., : -rope_dim], x[..., -rope_dim :]
    z = torch.view_as_complex(upcast(x_rot).unflatten(-1, (-1, 2)).contiguous())
    if inverse:
        freqs_cis = freqs_cis.conj()
    
    if z.ndim == 4:
        freqs_cis = freqs_cis.unsqueeze(1)
    
    x_rot = torch.view_as_real(z * freqs_cis).flatten(-2).to(x.dtype)
    return torch.cat([x_pass, x_rot], dim = -1)

In [ ]:
@lru_cache(8)
def hadamard_matrix(n : int, dtype = torch.float32, device = None):
    assert n & (n - 1) == 0
    with torch.inference_mode(False):
        h = torch.ones(1, 1, dtype = dtype, device = device)
        while h.shape[0] < n:
            h = torch.cat([torch.cat([h, h], dim = 1), torch.cat([h, -h], dim = 1)], dim = 0)
        return h / math.sqrt(n)


In [ ]:
def rotate_activation(x : torch.Tensor):
    xup = upcast(x)
    h = hadamard_matrix(x.shape[-1], xup.dtype, x.device)
    return (xup @ h).to(x.dtype)

In [ ]:
def pow2_ceil(x : torch.Tensor) -> torch.Tensor:
    mantissa, exponent = torch.frexp(x)
    return torch.ldexp(torch.ones_like(x), exponent - (mantissa == 0.5).int())

def fp8_simulate(x : torch.Tensor, block : int = 64) -> torch.Tensor:
    blocks = upcast(x).unflatten(-1, (-1, block)) # [..., d // block, block]
    amax = blocks.abs().amax(dim = -1, keepdim = True).clamp(min = 1e-4) # [..., d // block, 1]
    scale = pow2_ceil(amax * (1 / 448)) # [..., d // block, 1]
    q = (blocks / scale).clamp(-448, 448).to(torch.float8_e4m3fn) # [..., d // block, block]
    return (q.to(blocks.dtype) * scale).flatten(-2).to(x.dtype) # [..., d]

def fp4_simulate(x : torch.Tensor, block : int = 32) -> torch.Tensor:
    blocks = upcast(x).unflatten(-1, (-1, block)) # [..., d // block, block]
    amax = blocks.abs().amax(dim = -1, keepdim = True).clamp(min = 6 * 2 ** -126) # [..., d // block, 1]
    scale = pow2_ceil(amax * (1 / 6)) # [..., d // block, 1]
    v = (blocks / scale).clamp(-6, 6) # [..., d // block, block]
    step = torch.where(v.abs() < 2, 0.5, torch.where(v.abs() < 4, 1.0, 2.0)) # [..., d // block, block]
    return (torch.round(v / step) * step * scale).flatten(-2).to(x.dtype) # [..., d]

In [ ]:
def sparse_attn(
    q : torch.Tensor, # [b, s, h, d]
    kv : torch.Tensor, # [b, n, d]
    attn_sink : torch.Tensor, # [h]
    topk_idxs : torch.Tensor, # [b, s, k]
    softmax_scale : float,
    chunk : int = 256
):
    b, s = q.shape[: 2]
    batch = torch.arange(b, device = kv.device).view(b, 1, 1)
    out = torch.empty_like(q)

    for i in range(0, s, chunk):
        seleted_idxs = topk_idxs[:, i : i + chunk] # [b, c, k]
        valid = (seleted_idxs >= 0)
        keys = upcast(kv[batch, seleted_idxs.clamp(min = 0)]) # [b, c, k, d]
        logits = torch.einsum("bchd,bckd->bchk", upcast(q[:, i : i + chunk]), keys) * softmax_scale # [b, c, h, k]
        logits = logits.masked_fill_(~valid.unsqueeze(2), float("-inf"))
        sink = attn_sink.to(logits.dtype).view(1, 1, -1, 1).expand(*logits.shape[:-1], 1)
        probs = torch.cat([logits, sink], dim = -1).softmax(dim = -1)[..., :-1]
        out[:, i : i + chunk] = torch.einsum("bchk,bckd->bchd", probs, keys).to(q.dtype)
    return out

In [ ]:
@dataclass
class CompressorState:
    entries : torch.Tensor # [b, seq_len // ratio, d]
    kv : torch.Tensor # [b, coff * ratio, coff * d]
    score : torch.Tensor # [b, coff * ratio, coff * d]

@dataclass
class LayerState:
    window : torch.Tensor # [b, w, dh]
    compressor : CompressorState | None = None
    indexer : CompressorState | None = None

def compressor_state(b : int, ratio : int, d : int, max_seq_len : int, dtype, device):
    coff = 2 if ratio == 4 else 1
    raw_dtype = torch.promote_types(dtype, torch.float32)
    return CompressorState(
        entries = torch.zeros(b, max_seq_len // ratio, d, dtype = dtype, device = device),
        kv = torch.zeros(b, coff * ratio, coff * d, dtype = raw_dtype, device = device),
        score = torch.full((b, coff * ratio, coff * d), float("-inf"), dtype = raw_dtype, device = device)
    )

class SequenceState:
    def __init__(self, args : ModelArgs, b : int, dtype = torch.float32, device = None):
        self.pos = 0
        self.layers = []

        for r in args.compress_ratios[: args.n_layers]:
            layer = LayerState(torch.zeros(b, args.window_size, args.head_dim, dtype = dtype, device = device))
            if r:
                layer.compressor = compressor_state(b, r, args.head_dim, args.max_seq_len, dtype, device)
            if r == 4:
                layer.indexer = compressor_state(b, r, args.index_head_dim, args.max_seq_len, dtype, device)
            self.layers.append(layer)
      
        n_mtp = args.n_mtp_layers if args.dspark_block_size else 0
        self.mtp = [LayerState(torch.zeros(b, args.window_size, args.head_dim, dtype = dtype, device = device)) for _ in range(n_mtp)]

    

    def tensors(self) -> list[torch.Tensor]:
        out = []
        for layer in self.layers + self.mtp:
            out.append(layer.window) # [b, w, dh]
            for c in (layer.compressor, layer.indexer):
                if c is not None:
                    out += [c.entries, c.kv, c.score] # [b, max_seq_len // r, d], [b, coff * r, coff * d], [b, coff * r, coff * d]
        return out
    
    def clone(self) -> "SequenceState":
        return copy.deepcopy(self)
    
    @torch.inference_mode()
    def restore(self, snap : "SequenceState"):
        for t, saved in zip(self.tensors(), snap.tensors()):
            t.copy_(saved)
        self.pos = snap.pos

In [ ]:
class Compressor(nn.Module):
    def __init__(self, args : ModelArgs, compress_ratio : int, head_dim : int, rotate : bool = False):
        super().__init__()
        self.ratio = compress_ratio
        self.head_dim = head_dim
        self.rope_head_dim = args.rope_head_dim
        self.overlap = (compress_ratio == 4)
        self.rotate = rotate
        coff = 2 if self.overlap else 1
        self.wkv = nn.Linear(args.dim, coff * head_dim, bias = False)
        self.wgate = nn.Linear(args.dim, coff * head_dim, bias = False)
        self.ape = nn.Parameter(torch.randn(compress_ratio, coff * head_dim) * 0.1) # learner bias within block
        self.norm = RMSNorm(head_dim, args.norm_eps)
    
    def overlap_transform(self, t : torch.Tensor, fill : float) -> torch.Tensor:
        b, n, r, _ = t.shape
        d = self.head_dim
        out = t.new_full((b, n, 2 * r, d), fill)
        out[:, :, r :] = t[:, :, :, d :]
        out[:, 1 :, : r] = t[:, : -1, :, : d] # block shift
        return out

    
    def forward(self, x : torch.Tensor, freqs_cis : torch.Tensor, state : CompressorState | None = None):
        b, s, _ = x.shape
        r = self.ratio
        dtype = x.dtype
        x = upcast(x)
        cutoff = s - s % r
        kv = self.wkv(x) # [b, s, coff * dh]
        score = self.wgate(x) # [b, s, coff * dh]

        if state is not None:
            if self.overlap and cutoff >= r:
                state.kv[:, : r] = kv[:, cutoff - r : cutoff] # [b, r, coff * d]
                state.score[:, : r] = score[:, cutoff - r : cutoff] + self.ape # [b, r, coff * d]
            
            start = r if self.overlap else 0
            state.kv[: , start : start + s - cutoff] = kv[:, cutoff :] # [b, s - cutoff, coff * d]
            state.score[:, start : start + s - cutoff] = score[:, cutoff :] + self.ape[: s - cutoff] # [b, s - cutoff, coff * d]
        

        kv = kv[:, : cutoff].unflatten(1, (-1, r)) # [b, cutoff // r, r, coff * d]
        score = score[:, : cutoff].unflatten(1, (-1, r)) + self.ape # [b, cutoff // r, r, coff * d]

        if self.overlap:
            kv = self.overlap_transform(kv, 0.0) # [b, cutoff // r, coff * r, d]
            score = self.overlap_transform(score, float("-inf")) # [b, cutoff // r, coff * r, d]
        
        kv = (kv * score.softmax(dim = 2)).sum(dim = 2) # [b, cutoff // r, d]
        kv = self.norm(kv.to(dtype)) # [b, cutoff // r, d]
        kv = apply_rope(kv, freqs_cis[: cutoff : r]) # [b, cutoff // r, d]

        if self.rotate:
            kv = fp4_simulate(rotate_activation(kv)) # [b, cutoff // r, d]
        else:
            kv[..., : -self.rope_head_dim] = fp8_simulate(kv[..., : -self.rope_head_dim]) # [b, cutoff // r, d - rd]


        if state is not None:
            state.entries[:, : cutoff // r] = kv # [b, cutoff // r, d]

        return kv # [b, cutoff // r, d]
    
    def decode(self, x : torch.Tensor, freqs_cis : torch.Tensor, state : CompressorState, pos : int):
        # x : [b, 1, d]
        r, d = self.ratio, self.head_dim
        dtype = x.dtype
        x = upcast(x)
        row = (r if self.overlap else 0) + pos % r
        state.kv[:, row] = self.wkv(x)[:, 0] # [b, coff * d]
        state.score[:, row] = self.wgate(x)[:, 0] + self.ape[pos % r] # [b, coff * d]

        if (pos + 1) % r != 0:
            return None
        
        if self.overlap:
            kv = torch.cat([state.kv[:, : r, : d], state.kv[:, r :, d :]], dim = 1) # [b, 2 * r, d]
            score = torch.cat([state.score[:, : r, : d], state.score[:, r :, d :]], dim = 1) # [b, 2 * r, d]
        else:
            kv, score = state.kv, state.score # [b, r, d]
        
        kv = (kv * score.softmax(dim = 1)).sum(dim = 1, keepdim = True) # [b, 1, d]
        if self.overlap:
            state.kv[:, : r] = state.kv[:, r :]
            state.score[:, : r] = state.score[:, r :]
        
        kv = self.norm(kv.to(dtype)) # [b, 1, d]
        kv = apply_rope(kv, freqs_cis[pos + 1 - r].unsqueeze(0)) # [b, 1, d]
        if self.rotate:
            kv = fp4_simulate(rotate_activation(kv)) # [b, 1, d]
        else:
            kv[..., : -self.rope_head_dim] = fp8_simulate(kv[..., : -self.rope_head_dim]) # [b, 1, d]

        state.entries[:, pos // r] = kv[:, 0] # [b, d]
        return kv # [b, 1, d]
    
    def extend(self, x : torch.Tensor, freqs_cis : torch.Tensor, state : CompressorState, start_pos : int):
        b, s, _ = x.shape
        r = self.ratio
        dtype = x.dtype
        x = upcast(x)
        base = r if self.overlap else 0
        stored = base + start_pos % r
        pos = start_pos + torch.arange(s, device = x.device)
        kv = torch.cat([state.kv[:, : stored], self.wkv(x)], dim = 1) # [b, stored + s, coff * d]
        score = torch.cat([state.score[:, : stored], self.wgate(x) + self.ape[pos % r]], dim = 1) # [b, stored + s, coff * d]
        keep = torch.arange(max(0, s - r), s, device = x.device) # [min(s, r)]
        state.kv[:, base + pos[keep] % r] = kv[:, stored + keep] # [b, min(s, r), coff * d]
        state.score[:, base + pos[keep] % r] = score[:, stored + keep] # [b, min(s, r), coff * d]
        first, last = start_pos // r, (start_pos + s) // r

        if first == last:
            return None

        cutoff = kv.shape[1] - kv.shape[1] % r
        kv = kv[:, : cutoff].unflatten(1, (-1, r)) # [b, cutoff // r, r, coff * d]
        score = score[:, : cutoff].unflatten(1, (-1, r)) # [b, cutoff // r, r, coff * d]

        if self.overlap:
            state.kv[:, : r] = kv[:, -1] # [b, r, 2 * d]
            state.score[:, : r] = score[:, -1] # [b, r, 2 * d]
            kv = self.overlap_transform(kv, 0.0)[:, 1 :] # [b, last - first, 2 * r, d]
            score = self.overlap_transform(score, float("-inf"))[:, 1 :] # [b, last - first, 2 * r, d]

        kv = (kv * score.softmax(dim = 2)).sum(dim = 2) # [b, last - first, d]
        kv = self.norm(kv.to(dtype)) # [b, last - first, d]
        kv = apply_rope(kv, freqs_cis[first * r : last * r : r]) # [b, last - first, d]
        if self.rotate:
            kv = fp4_simulate(rotate_activation(kv)) # [b, last - first, d]
        else:
            kv[..., : -self.rope_head_dim] = fp8_simulate(kv[..., : -self.rope_head_dim]) # [b, last - first, d]

        state.entries[:, first : last] = kv # [b, last - first, d]
        return kv # [b, last - first, d]


In [ ]:
class Indexer(nn.Module):
    def __init__(self, args : ModelArgs, compress_ratio : int = 4):
        super().__init__()
        self.n_heads = args.index_n_heads
        self.head_dim = args.index_head_dim
        self.index_topk = args.index_topk
        self.ratio = compress_ratio
        self.wq_b = nn.Linear(args.q_lora_rank, self.n_heads * self.head_dim, bias = False)
        self.weights_proj = nn.Linear(args.dim, self.n_heads, bias = False)
        self.softmax_scale = self.head_dim ** -0.5
        self.compressor = Compressor(args, compress_ratio, self.head_dim, rotate = True)
    
    def forward(
        self, 
        x : torch.Tensor, 
        qr : torch.Tensor, 
        freqs_cis : torch.Tensor, 
        offset : int, 
        state : CompressorState | None = None,
        start_pos : int = 0
    ) -> torch.Tensor:
        # x : [b, s, dim] - prefill_part
        # qr : [b, s, q_lora_rank]
        b, s, _ = x.shape
        
        q = self.wq_b(qr).unflatten(-1, (self.n_heads, self.head_dim)) # [b, s, nh, d]
        q = fp4_simulate(rotate_activation(apply_rope(q, freqs_cis[start_pos : start_pos + s]))) # [b, s, nh, d]


        if start_pos > 0:
            self.compressor.extend(x, freqs_cis, state, start_pos)
            k = state.entries[:, : (start_pos + s) // self.ratio] # [b, n, d]
        else:
            k = self.compressor(x, freqs_cis, state) # [b, n, d]
    
        w = self.weights_proj(x) * (self.softmax_scale * (self.n_heads ** -0.5)) # [b, s, nh]
        score = torch.einsum("bshd,bnd->bshn", q, k).relu() # [b, s, nh, n]
        score = (score * w.unsqueeze(-1)).sum(dim = 2) # [b, s, n]

        

        n_visible = (start_pos + torch.arange(1, s + 1, device = x.device)) // self.ratio
        visible = torch.arange(k.shape[1], device = x.device) < n_visible.unsqueeze(1) # [s, n]
        score = score.masked_fill(~visible, float("-inf"))
        topk = score.topk(min(self.index_topk, k.shape[1]), dim = -1).indices
        return torch.where(topk < n_visible.unsqueeze(1), topk + offset, -1)

In [ ]:
def window_topk_idxs(window_size : int, s : int, device = None) -> torch.Tensor:
    # prefill windows idxs
    t = torch.arange(s, device = device).unsqueeze(1) # [s, 1]
    idx = (t - window_size + 1).clamp(min = 0) + torch.arange(min(s, window_size), device = device) # [s, w]
    return torch.where(idx > t, -1, idx)

def compress_topk_idxs(ratio : int, s : int, offset : int, device = None, start_pos : int = 0) -> torch.Tensor:
    # for HCA prefill
    blocks = torch.arange((start_pos + s) // ratio, device = device)
    n_visible = (start_pos + torch.arange(1, s + 1, device = device)).unsqueeze(1) // ratio
    return torch.where(blocks < n_visible, blocks + offset, -1)

def window_decode_idxs(window_size : int, pos : int, device = None):
    if pos > window_size - 1:
        return torch.arange(pos + 1, pos + 1 + window_size, device = device) % window_size
    slots = torch.arange(window_size, device = device)
    return torch.where(slots <= pos, slots, -1)

def window_extend_idxs(window_size : int, start_pos : int, s : int, device = None) -> torch.Tensor:
    t = start_pos + torch.arange(s, device = device).unsqueeze(1) # [s, 1]
    pos = (t - window_size + 1).clamp(min = 0) + torch.arange(window_size, device = device) # [s, w]
    rows = torch.where(pos < start_pos, pos % window_size, window_size + pos - start_pos)
    return torch.where(pos > t, -1, rows) # [s, w]

In [ ]:
class Attention(nn.Module):
    def __init__(self, layer_id : int, args : ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.head_dim = args.head_dim
        self.rope_head_dim = args.rope_head_dim
        self.n_groups = args.o_groups
        self.o_lora_rank = args.o_lora_rank
        self.window_size = args.window_size
        self.eps = args.norm_eps
        self.compress_ratio = args.compress_ratios[layer_id]
        self.softmax_scale = self.head_dim ** -0.5

        self.attn_sink = nn.Parameter(torch.randn(self.n_heads))
        self.wq_a = nn.Linear(args.dim, args.q_lora_rank, bias = False)
        self.q_norm = RMSNorm(args.q_lora_rank, self.eps)
        self.wq_b = nn.Linear(args.q_lora_rank, self.n_heads * self.head_dim, bias = False)
        self.wkv = nn.Linear(args.dim, self.head_dim, bias = False)
        self.kv_norm = RMSNorm(self.head_dim, self.eps)

        self.wo_a = nn.Linear(self.n_heads * self.head_dim // self.n_groups, self.n_groups * self.o_lora_rank, bias = False)
        self.wo_b = nn.Linear(self.n_groups * self.o_lora_rank, args.dim, bias = False)
        if self.compress_ratio:
            self.compressor = Compressor(args, self.compress_ratio, self.head_dim)
            self.indexer = Indexer(args, self.compress_ratio) if self.compress_ratio == 4 else None
        
        self.register_buffer("freqs_cis", layer_freqs_cis(args, self.compress_ratio), persistent = False)
    

    def forward(self, x : torch.Tensor, state : LayerState | None = None, start_pos : int = 0):
        # x : [b, s, d] - prefill stage
        b, s, _ = x.shape
        freqs = self.freqs_cis[start_pos : start_pos + s] # [s, rope_dim // 2]

        qr = self.q_norm(self.wq_a(x)) # [b, s, q_lora_rank]
        q = self.wq_b(qr).unflatten(-1, (self.n_heads, self.head_dim)) # [b, s, h, c]
        q = q * torch.rsqrt(q.square().mean(-1, keepdim = True) + self.eps)
        q = apply_rope(q, freqs) # [b, s, h, c]

        kv = apply_rope(self.kv_norm(self.wkv(x)), freqs) # [b, s, c]
        kv[..., : -self.rope_head_dim] = fp8_simulate(kv[..., : -self.rope_head_dim])
        
        if start_pos == 0:
            idxs = window_topk_idxs(self.window_size, s, x.device).expand(b, -1, -1) # [b, s, w]

            if state is not None:
                keep = torch.arange(max(0, s - self.window_size), s, device = x.device) # [min(s, w)]
                state.window[:, keep % self.window_size] = kv[:, keep] # [b, min(s, w), c]

            if self.compress_ratio:
                comp_state = state.compressor if state is not None else None
                index_state = state.indexer if state is not None else None
                if self.indexer is not None:
                    comp_idxs = self.indexer(x, qr, self.freqs_cis, offset = s, state = index_state) # [b, s, k]
                else:
                    comp_idxs = compress_topk_idxs(self.compress_ratio, s, s, x.device).expand(b, -1, -1) # [b, s, k]

                idxs = torch.cat([idxs, comp_idxs], dim = -1) # [b, s, w + k]
                kv = torch.cat([kv, self.compressor(x, self.freqs_cis, comp_state)], dim = 1) # [b, s + n, c]
        else:
            assert state is not None
            w = self.window_size
            idxs = window_extend_idxs(w, start_pos, s, x.device).expand(b, -1, -1) # [b, s, w]
            if self.compress_ratio:
                n = (start_pos + s) // self.compress_ratio
                if self.indexer is not None:
                    comp_idxs = self.indexer(x, qr, self.freqs_cis, offset = w + s, state = state.indexer, start_pos = start_pos) # [b, s, k]
                else:
                    comp_idxs = compress_topk_idxs(self.compress_ratio, s, w + s, x.device, start_pos).expand(b, -1, -1) # [b, s, n]

                idxs = torch.cat([idxs, comp_idxs], dim = -1) # [b, s, w + k]
            
            kv = torch.cat([state.window, kv], dim = 1) # [b, w + s, c]
            keep = torch.arange(max(0, s - w), s, device = x.device) # [min(s, w)]
            state.window[:, (start_pos + keep) % w] = kv[:, w + keep] # [b, min(s, w), c]
            if self.compress_ratio:
                self.compressor.extend(x, self.freqs_cis, state.compressor, start_pos)
                kv = torch.cat([kv, state.compressor.entries[:, : n]], dim = 1) # [b, w + s + n, c]


        o = sparse_attn(q, kv, self.attn_sink, idxs, self.softmax_scale) # [b, s, h, c]
        o = apply_rope(o, freqs, inverse = True) # [b, s, h, c]

        o = o.reshape(b, s, self.n_groups, -1) # [b, s, g, h * c / g]
        wo_a = self.wo_a.weight.view(self.n_groups, self.o_lora_rank, -1) # [g, r, h * c / g]
        o = torch.einsum("bsgd,grd->bsgr", o, wo_a) # [b, s, g, r]
        return self.wo_b(o.flatten(2)) # [b, s, d]

In [ ]:
def hc_split_sinkhorn(
    mixes : torch.Tensor,
    hc_scale : torch.Tensor,
    hc_base : torch.Tensor,
    hc_mult : int = 4,
    sinkhorn_iters : int = 20,
    eps : float = 1e-6
):  
    # mixes : [b, s, hc * (2 + hc)]
    hc = hc_mult
    pre, post, comb = mixes.split([hc, hc, hc * hc], dim = -1) # [b, s, hc], [b, s, hc], [b, s, hc * hc]
    base_pre, base_post, base_comb = hc_base.split([hc, hc, hc * hc])
    pre = torch.sigmoid(pre * hc_scale[0] + base_pre) + eps
    post = 2 * torch.sigmoid(post * hc_scale[1] + base_post)
    comb = (comb * hc_scale[2] + base_comb).unflatten(-1, (hc, hc))

    comb = comb.softmax(dim = -1) + eps
    comb = comb / (comb.sum(dim = -2, keepdim = True) + eps)

    for _ in range(sinkhorn_iters - 1):
        comb = comb / (comb.sum(dim = -1, keepdim = True) + eps)
        comb = comb / (comb.sum(dim = -2, keepdim = True) + eps)
    
    return pre, post, comb


def hc_pre(
    x : torch.Tensor, # [b, s, hc, d]
    hc_fn : torch.Tensor,  # [hc * (2 + hc), hc * d]
    hc_scale : torch.Tensor, # [3, ]
    hc_base : torch.Tensor, # [hc * (2 + hc), ]
    norm_eps : float,
    hc_mult : int = 4,
    sinkhorn_iters : int = 20,
    hc_eps : float = 1e-6
):
    shape, dtype = x.shape, x.dtype
    x = upcast(x.flatten(2)) # [b, s, hc * d]
    rsqrt = torch.rsqrt(x.square().mean(-1, keepdim = True) + norm_eps)
    mixes = F.linear(x, hc_fn) * rsqrt # [b, s, hc * (2 + hc)]

    pre, post, comb = hc_split_sinkhorn(mixes, hc_scale, hc_base, hc_mult, sinkhorn_iters, hc_eps)
    # pre : [b, s, hc], post : [b, s, hc], comb : [b, s, hc, hc]
    y = (pre.unsqueeze(-1) * x.view(shape)).sum(dim = 2)
    return y.to(dtype), post, comb # [b, s, d], [b, s, hc], [b, s, hc, hc]

def hc_post(
    x : torch.Tensor,  # [b, s, d] 
    residual : torch.Tensor,  # [b, s, hc, d]
    post : torch.Tensor,  # [b, s, hc]
    comb : torch.Tensor) -> torch.Tensor: # [b, s, hc, hc]
    y = ((residual.unsqueeze(-2) * comb.unsqueeze(-1)).sum(dim = 2)) + (post.unsqueeze(-1) * x.unsqueeze(-2))
    return y.type_as(x) # [b, s, hc, d]

In [ ]:
class Gate(nn.Module):
    def __init__(self, layer_id : int, args : ModelArgs):
        super().__init__()
        self.topk = args.n_activated_experts
        self.route_scale = args.route_scale
        self.hash = layer_id < args.n_hash_layers
        self.weight = nn.Parameter(torch.randn(args.n_routed_experts, args.dim) * (args.dim ** -0.5))

        if self.hash:
            table = torch.randn(args.vocab_size, args.n_routed_experts).argsort(dim = -1)[:, : self.topk]
            self.register_buffer("tid2eid", table.int())
            self.bias = None
        else:
            self.bias = nn.Parameter(torch.randn(args.n_routed_experts) * 0.1)
        
    def forward(self, x : torch.Tensor, input_ids : torch.Tensor):
        # x : [b * c, dim], input_ids : [b * c, ]
        scores = F.softplus(F.linear(upcast(x), upcast(self.weight))).sqrt() # [b * c, E]
        if self.hash:
            indices = self.tid2eid[input_ids].long() # [b * c, topk]
        else:
            indices = (scores + self.bias).topk(self.topk, dim = -1).indices # [b * c, topk]
        
        weights = scores.gather(1, indices) # [b * c, topk]
        weights = weights / weights.sum(dim = -1, keepdim = True) * self.route_scale
        return weights, indices

In [ ]:
class Expert(nn.Module):
    def __init__(
        self,
        dim : int,
        inter_dim : int,
        swiglu_limit : float = 0.0
    ):
        super().__init__()
        self.w1 = nn.Linear(dim, inter_dim, bias = False)
        self.w2 = nn.Linear(inter_dim, dim, bias = False)
        self.w3 = nn.Linear(dim, inter_dim, bias = False)
        self.swiglu_limit = swiglu_limit
    
    def forward(
        self,
        x : torch.Tensor,
        weights : torch.Tensor | None = None
    ):
        dtype = x.dtype
        gate = upcast(self.w1(x))
        up = upcast(self.w3(x))

        if self.swiglu_limit > 0:
            up = up.clamp(-self.swiglu_limit, self.swiglu_limit)
            gate = gate.clamp(max = self.swiglu_limit) # For low, as we use silu x * sigmoid(x), so already -> 0
        
        h = F.silu(gate) * up
        if weights is not None:
            h = weights * h
        
        return self.w2(h.to(dtype))

In [ ]:
class MoE(nn.Module):
    def __init__(
        self,
        layer_id : int,
        args : ModelArgs
    ):
        super().__init__()
        self.dim = args.dim
        self.n_routed_experts = args.n_routed_experts
        self.gate = Gate(layer_id, args)
        self.experts = nn.ModuleList(
            [
                Expert(self.dim, args.moe_inter_dim, args.swiglu_limit)
                for _ in range(args.n_routed_experts)
            ]
        )
        self.shared_experts = Expert(self.dim, args.moe_inter_dim, args.swiglu_limit)

    
    def forward(self, x : torch.Tensor, input_ids : torch.Tensor):
        # x : [b, s, d], input_ids : [b, s]
        shape = x.shape
        x = x.view(-1, self.dim) # [b * s, d]
        weights, indices = self.gate(x, input_ids.flatten()) # [b * s, k], [b * s, k]
        y = torch.zeros_like(x, dtype = torch.promote_types(x.dtype, torch.float32))

        flat = indices.flatten() # [b * s * k]
        order = flat.argsort(stable = True) # [b * s * k]
        counts = torch.zeros(self.n_routed_experts, dtype = torch.long, device = x.device).scatter_add_(0, flat, torch.ones_like(flat)) # [E]
        counts = counts.tolist()

        k = indices.shape[1]
        start = 0
        for i, expert in enumerate(self.experts):
            if counts[i] == 0:
                continue
            rows = order[start : start + counts[i]] # [n_i]
            start += counts[i]
            token_idxs, slots = rows // k, rows % k # [n_i], [n_i]
            y[token_idxs] += expert(x[token_idxs], weights[token_idxs, slots, None]) # [n_i, d]

        y += self.shared_experts(x) # shared expert has wt 1.
        return y.type_as(x).view(shape)



In [ ]:
class Block(nn.Module):
    attention_cls = Attention
    
    def __init__(self, layer_id : int, args : ModelArgs):
        super().__init__()
        self.norm_eps = args.norm_eps
        self.hc_mult = args.hc_mult
        self.hc_sinkhorn_iters = args.hc_sinkhorn_iters
        self.hc_eps = args.hc_eps

        self.attn = self.attention_cls(layer_id, args)
        self.attn_norm = RMSNorm(args.dim, args.norm_eps)
        self.ffn = MoE(layer_id, args)
        self.ffn_norm = RMSNorm(args.dim, args.norm_eps)

        mix_hc = (2 + args.hc_mult) * args.hc_mult
        hc_dim = args.hc_mult * args.dim

        self.hc_attn_fn = nn.Parameter(torch.randn(mix_hc, hc_dim) * 0.02)
        self.hc_attn_base = nn.Parameter(torch.randn(mix_hc) * 0.5)
        self.hc_attn_scale = nn.Parameter(torch.randn(3) * 0.5 + 1)

        self.hc_ffn_fn = nn.Parameter(torch.randn(mix_hc, hc_dim) * 0.02)
        self.hc_ffn_base = nn.Parameter(torch.randn(mix_hc) * 0.5)
        self.hc_ffn_scale = nn.Parameter(torch.randn(3) * 0.5 + 1)
        
    
    def forward(self, x : torch.Tensor, input_ids : torch.Tensor, state : LayerState | None = None, start_pos : int = 0, *attn_args):
        # x : [b, s, hc, d], input_ids : [b, s]
        res = x
        x, post, comb = hc_pre(
            x, 
            self.hc_attn_fn, 
            self.hc_attn_scale, 
            self.hc_attn_base, 
            self.norm_eps, 
            self.hc_mult, 
            self.hc_sinkhorn_iters, 
            self.hc_eps
        )
        x = self.attn(self.attn_norm(x), state, start_pos, *attn_args)
        x = hc_post(x, res, post, comb)

        res = x
        x, post, comb = hc_pre(
            x, 
            self.hc_ffn_fn, 
            self.hc_ffn_scale, 
            self.hc_ffn_base, 
            self.norm_eps, 
            self.hc_mult, 
            self.hc_sinkhorn_iters, 
            self.hc_eps
        )
        x = self.ffn(self.ffn_norm(x), input_ids)
        return hc_post(x, res, post, comb)

In [ ]:
def hc_head(
    x : torch.Tensor,
    hc_fn : torch.Tensor,
    hc_scale : torch.Tensor,
    hc_base : torch.Tensor,
    norm_eps : float,
    hc_eps : float
):
    # merge all hc streams into use, like hc pre 
    # x : [b, s, h, d]
    shape, dtype = x.shape, x.dtype
    x = upcast(x.flatten(2)) # [b, s, h * d]
    rsqrt = torch.rsqrt(x.square().mean(-1, keepdim = True) + norm_eps)
    mixes = F.linear(x, hc_fn) * rsqrt # [b, s, h]
    pre = torch.sigmoid(mixes * hc_scale + hc_base) + hc_eps # [b, s, h]
    y = (pre.unsqueeze(-1) * x.view(shape)).sum(dim = 2) # [b, s, d]
    return y.to(dtype)

In [ ]:
def sample(logits : torch.Tensor, temperature : float = 1.0, generator : torch.Generator | None = None) -> torch.Tensor:
    # logits : [b, s, V]
    if temperature == 0.0:
        return logits.argmax(dim = -1) # [b, s]
   
    probs = torch.softmax(logits / max(temperature, 1e-5), dim = -1, dtype = torch.float32)
    noise = torch.empty_like(probs).exponential_(1, generator = generator)
    return probs.div_(noise).argmax(dim = -1) # [b, s]

In [ ]:
class DSparkMarkovHead(nn.Module):
    def __init__(self, vocab_size : int, rank : int):
        super().__init__()
        self.markov_w1 = nn.Embedding(vocab_size, rank)
        self.markov_w2 = nn.Linear(rank, vocab_size, bias = False)
    
    def forward(self, ids : torch.Tensor):
        emb = self.markov_w1(ids)
        return self.markov_w2(upcast(emb)), emb # [b, vocab_size], [b, r]
    

class DSparkConfidenceHead(nn.Module):
    def __init__(self, dim : int):
        super().__init__()
        self.proj = nn.Linear(dim, 1, bias = False)
    
    def forward(self, x : torch.Tensor, markov : torch.Tensor):
        # x : [b, k, d], markov : [b, k, d]
        return self.proj(upcast(torch.cat([x, markov], dim = -1))).squeeze(-1) # [b, k]

In [ ]:
class DSparkAttention(Attention):
    def forward(self, x : torch.Tensor, state : LayerState, start_pos : int, main_x : torch.Tensor):
        # x : [b, k, d], main_x : [b, s, d]
        b, s, _ = main_x.shape
        w = self.window_size
        main_kv = apply_rope(self.kv_norm(self.wkv(main_x)), self.freqs_cis[start_pos : start_pos + s])
        main_kv[..., : -self.rope_head_dim] = fp8_simulate(main_kv[..., : -self.rope_head_dim])
        keep = torch.arange(max(0, s - w), s, device = x.device) # [min(s, w)]
        state.window[:, (start_pos + keep) % w] = main_kv[:, keep] # [b, min(s, w), c]
        if start_pos == 0:
            return x
        
        k = x.shape[1]
        freqs = self.freqs_cis[start_pos + s : start_pos + s + k]
        q = self.wq_b(self.q_norm(self.wq_a(x))).unflatten(-1, (self.n_heads, self.head_dim)) # [b, k, h, c]
        q = q * torch.rsqrt(q.square().mean(-1, keepdim = True) + self.eps)
        q = apply_rope(q, freqs) # [b, k, h, c]
        kv = apply_rope(self.kv_norm(self.wkv(x)), freqs) # [b, k, c]
        kv[..., : -self.rope_head_dim] = fp8_simulate(kv[..., : -self.rope_head_dim])

        n = min(w, start_pos + s)
        idxs = torch.cat([torch.arange(n, device = x.device), w + torch.arange(k, device = x.device)]).expand(b, k, -1) # [b, k, n + k]
        kv = torch.cat([state.window, kv], dim = 1) # [b, w + k, c]
        o = sparse_attn(q, kv, self.attn_sink, idxs, self.softmax_scale) # [b, k, h, c]
        o = apply_rope(o, freqs, inverse = True) # [b, k, h, c]

        o = o.reshape(b, k, self.n_groups, -1) # [b, k, g, h * c / g]
        wo_a = self.wo_a.weight.view(self.n_groups, self.o_lora_rank, -1) # [g, r, h * c / g]
        o = torch.einsum("bsgd,grd->bsgr", o, wo_a) # [b, k, g, r]
        return self.wo_b(o.flatten(2)) # [b, k, d]


In [ ]:
class DSparkBlock(Block):
    attention_cls = DSparkAttention

    def __init__(self, layer_id : int, args : ModelArgs):
        super().__init__(layer_id, args)
        stage = layer_id - args.n_layers
        self.block_size = args.dspark_block_size
        self.noise_token_id = args.dspark_noise_token_id
        if stage == 0:
            self.main_proj = nn.Linear(len(args.dspark_target_layer_ids) * args.dim, args.dim, bias = False) # [d, t * d]
            self.main_norm = RMSNorm(args.dim, args.norm_eps)
        if stage == args.n_mtp_layers - 1:
            self.norm = RMSNorm(args.dim, args.norm_eps)
            self.markov_head = DSparkMarkovHead(args.vocab_size, args.dspark_markov_rank)
            self.confidence_head = DSparkConfidenceHead(args.dim + args.dspark_markov_rank)
            hc_dim = args.hc_mult * args.dim
            self.hc_head_fn = nn.Parameter(torch.randn(args.hc_mult, hc_dim) * 0.02) # [hc, hc * d]
            self.hc_head_base = nn.Parameter(torch.randn(args.hc_mult) * 0.5) # [hc]
            self.hc_head_scale = nn.Parameter(torch.randn(1) * 0.5 + 1) # [1]

    def forward(self, x : torch.Tensor, input_ids : torch.Tensor, state : LayerState, start_pos : int, main_x : torch.Tensor):
        # x : [b, k, hc, d], main_x : [b, s, d]
        if start_pos == 0:
            return self.attn(x, state, start_pos, main_x)
        return super().forward(x, input_ids, state, start_pos, main_x) # [b, k, hc, d]

    def forward_embed(self, main_hidden : torch.Tensor, next_ids : torch.Tensor, embed : nn.Embedding):
        # main_hidden : [b, s, t * d], next_ids : [b]
        main_x = self.main_norm(self.main_proj(main_hidden)) # [b, s, d]
        draft_ids = next_ids.new_full((next_ids.shape[0], self.block_size), self.noise_token_id) # [b, k]
        draft_ids[:, 0] = next_ids
        x = embed(draft_ids).unsqueeze(2).repeat(1, 1, self.hc_mult, 1) # [b, k, hc, d]
        return x, main_x, draft_ids

    def forward_head(self, x : torch.Tensor, next_ids : torch.Tensor, head : nn.Linear, temperature : float, generator : torch.Generator | None):
        # x : [b, k, hc, d], next_ids : [b]
        x = hc_head(x, self.hc_head_fn, self.hc_head_scale, self.hc_head_base, self.norm_eps, self.hc_eps) # [b, k, d]
        logits = head(upcast(self.norm(x))) # [b, k, vocab]
        ids, markov = [next_ids], []
        for i in range(self.block_size):
            bias, emb = self.markov_head(ids[i]) # [b, vocab], [b, r]
            logits[:, i] += bias
            markov.append(emb)
            ids.append(sample(logits[:, i], temperature, generator)) # [b]
        confidence = self.confidence_head(x, torch.stack(markov, dim = 1)) # [b, k]
        return torch.stack(ids, dim = 1), logits, confidence # [b, k + 1], [b, k, vocab], [b, k]

In [ ]:
class Transformer(nn.Module):
    def __init__(self, args : ModelArgs):
        super().__init__()
        self.hc_mult = args.hc_mult
        self.hc_eps = args.hc_eps
        self.norm_eps = args.norm_eps
        self.target_layer_ids = args.dspark_target_layer_ids

        self.embed = nn.Embedding(args.vocab_size, args.dim) # [vocab, d]
        self.layers = nn.ModuleList([Block(layer_id, args) for layer_id in range(args.n_layers)])
        self.norm = RMSNorm(args.dim, args.norm_eps) # [d]
        self.head = nn.Linear(args.dim, args.vocab_size, bias = False) # [vocab, d]

        hc_dim = args.hc_mult * args.dim
        self.hc_head_fn = nn.Parameter(torch.randn(args.hc_mult, hc_dim) * 0.02) # [hc, hc * d]
        self.hc_head_base = nn.Parameter(torch.randn(args.hc_mult) * 0.5) # [hc]
        self.hc_head_scale = nn.Parameter(torch.randn(1) * 0.5 + 1) # [1]

        n_mtp = args.n_mtp_layers if args.dspark_block_size else 0
        self.mtp = nn.ModuleList([DSparkBlock(args.n_layers + i, args) for i in range(n_mtp)])


    @torch.inference_mode()
    def forward(self, input_ids : torch.Tensor, state : SequenceState | None = None, return_all_logits : bool = False, return_hidden : bool = False):
        # input_ids : [b, s]
        start_pos = state.pos if state is not None else 0
        h = self.embed(input_ids) # [b, s, d]
        h = h.unsqueeze(2).repeat(1, 1, self.hc_mult, 1) # [b, s, hc, d]
        
        layer_states = state.layers if state is not None else [None] * len(self.layers)
        hidden = []
        for i, (layer, layer_state) in enumerate(zip(self.layers, layer_states)):
            h = layer(h, input_ids, layer_state, start_pos) # [b, s, hc, d]
            if return_hidden and i in self.target_layer_ids:
                hidden.append(h.mean(dim = 2)) # [b, s, d]

        if state is not None:
            state.pos += input_ids.shape[1]

        h = hc_head(h, self.hc_head_fn, self.hc_head_scale, self.hc_head_base, self.norm_eps, self.hc_eps) # [b, s, d]
        h = self.norm(h) # [b, s, d]
        if not return_all_logits:
            h = h[:, -1] # [b, d]
        logits = self.head(upcast(h)) # [b, vocab] or [b, s, vocab]

        if return_hidden:
            return logits, torch.cat(hidden, dim = -1) # [b, s, t * d]
        
        return logits
    

    @torch.inference_mode()
    def forward_spec(self, next_ids : torch.Tensor, main_hidden : torch.Tensor, state : SequenceState, temperature : float = 1.0, generator : torch.Generator | None = None):
        # next_ids : [b], main_hidden : [b, s, t * d]
        start_pos = state.pos - main_hidden.shape[1]
        h, main_x, draft_ids = self.mtp[0].forward_embed(main_hidden, next_ids, self.embed)
        for layer, layer_state in zip(self.mtp, state.mtp):
            h = layer(h, draft_ids, layer_state, start_pos, main_x) # [b, k, hc, d]
        if start_pos == 0:
            return None
        return self.mtp[-1].forward_head(h, next_ids, self.head, temperature, generator)


In [ ]:
@torch.inference_mode()
def generate(
    model : Transformer, 
    args : ModelArgs, 
    prompts : list[list[int]] | torch.Tensor,
    n_new : int,
    temperature : float = 0.0,
    seed : int = 0,
    eos_id : int | None = None
):
    # prompts : list of token ids
    if isinstance(prompts, torch.Tensor):
        prompts = prompts.tolist()
    
    b, lens = len(prompts), [len(p) for p in prompts]
    assert max(lens) <= args.max_seq_len
    device = model.embed.weight.device

    tokens = torch.full((b, max(lens) + n_new), -1, dtype = torch.long, device = device) # [b, p_max + n_new]
    for i, p in enumerate[list[int]](prompts):
        tokens[i, : lens[i]] = torch.tensor(p, device = device)
    
    in_prompt = (tokens != -1) # [b, p_max + n_new]
    finished = torch.zeros(b, dtype = torch.bool, device = device) # [b]
    generator = torch.Generator(device).manual_seed(seed)
    state = SequenceState(args, b, device = device)

    for pos in range(min(lens), min(args.max_seq_len, tokens.shape[1])):
        logits = model(tokens[:, state.pos : pos], state) # [b, vocab]
        next_id = sample(logits, temperature, generator) # [b]
        tokens[:, pos] = torch.where(in_prompt[:, pos], tokens[:, pos], next_id)
        if eos_id is not None:
            finished |= ~in_prompt[:, pos] & (tokens[:, pos] == eos_id)
        if finished.all():
            break
    
    return torch.stack([tokens[i, n : n + n_new] for i, n in enumerate(lens)])

In [ ]:
import sys
sys.path.insert(0, "deepseek_official/0731/encoding/")

from tokenizers import Tokenizer
from encoding_dsv4 import encode_messages, parse_message_from_completion_text


class ChatTokenizer:
    def __init__(self, path : str):
        self.tokenizer = Tokenizer.from_file(path)
        self.eos_id = self.tokenizer.token_to_id("<｜end▁of▁sentence｜>")
    
    def encode_chat(self, messages : list[dict], thinking_mode : str = "chat") -> list[int]:
        return self.tokenizer.encode(encode_messages(messages, thinking_mode = thinking_mode)).ids

    def decode_completion(self, ids : list[int]) -> str:
        ids = [t for t in ids if t >= 0]
        if self.eos_id in ids:
            ids = ids[: ids.index(self.eos_id)]
        return self.tokenizer.decode(ids + [self.eos_id], skip_special_tokens = False)

In [ ]:
torch.manual_seed(0)
args = tiny_args()
model = Transformer(args)
keys = list(model.state_dict())
print([k for k in keys if not k.startswith('layers.')])
print([k for k in keys if k.startswith('layers.0.hc_')])
print(sum(p.numel() for p in model.parameters()), 'parameters')

b, s = 2, 37
ids = torch.randint(0, args.vocab_size, (b, s))
logits = model(ids)                                        # [b, vocab]: last position only, like official
all_logits = model(ids, return_all_logits=True)            # [b, s, vocab]
print(logits.shape, all_logits.shape, torch.allclose(logits, all_logits[:, -1], atol=1e-6))

ids2 = ids.clone()
ids2[0, 20] = (ids2[0, 20] + 1) % args.vocab_size          # change one token: sequence 0, position 20
diff = (model(ids2, return_all_logits=True) - all_logits).abs().amax(dim=-1)   # [b, s]: biggest change per position
print(bool((diff[0, :20] < 1e-5).all()), bool((diff[0, 20:] > 1e-4).all()), bool((diff[1] < 1e-5).all()))

model.double()                                             # same weights in float64 (in place)
all64 = model(ids, return_all_logits=True)
print(all64.dtype, ((all64 - all_logits.double()).abs().max() / all64.abs().max()).item())

In [ ]:
RUN_REAL_SIZE = True   # set False to skip this slow cell during Restart & Run All

if RUN_REAL_SIZE:
    import sys, time, resource
    sys.path.insert(0, "checks")
    from load_official import load_official, official_args

    off = load_official("april")                    # DeepSeek's model.py, running on the CPU
    args = ModelArgs(n_routed_experts = 8)          # real sizes, but 8 experts instead of 256
    oargs = official_args(off, args)                # the same settings, as the official ModelArgs

    for layer_id in (0, 2, 3):                      # window only + hash, CSA + hash, HCA + learned routing
        torch.manual_seed(layer_id)
        ours = Block(layer_id, args)
        theirs = off.Block(layer_id, oargs).float()
        theirs.load_state_dict(ours.state_dict(), strict = True)   # copy our random weights into theirs
        for s in (1, 127, 128, 129, 600, 2304):
            x = torch.randn(1, s, args.hc_mult, args.dim)          # [b, s, hc, d]: the 4 streams
            ids = torch.randint(0, args.vocab_size, (1, s))        # [b, s]
            with torch.inference_mode():
                t0 = time.perf_counter()
                y_ours = ours(x, ids)
                t1 = time.perf_counter()
                y_theirs = theirs(x, 0, ids)                       # 0 = start_pos: a prefill from position 0
            print(f"layer {layer_id}  s = {s:4d}  identical {torch.equal(y_ours, y_theirs)}  ours {t1 - t0:5.2f} s")
        del ours, theirs

    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    print(f"peak memory {peak / 2**30 if sys.platform == 'darwin' else peak / 2**20:.1f} GiB")


In [ ]:
import sys
sys.path.insert(0, "checks")
from load_official import load_official, official_args

off = load_official("april")
args = tiny_args()
w = args.window_size

for s in (1, 3, 4, 5, 8, 9, 16, 17, 37, 64):
    torch.manual_seed(s)
    b = 2
    model = Transformer(args)
    theirs = off.Transformer(official_args(off, args, max_batch_size = b)).float()
    theirs.load_state_dict(model.state_dict(), strict = True)
    ids = torch.randint(0, args.vocab_size, (b, s))

    state = SequenceState(args, b)
    same_logits = torch.equal(model(ids, state), theirs(ids, 0))

    pairs = []
    for ours, layer in zip(state.layers, theirs.layers):
        a = layer.attn
        pairs.append((ours.window, a.kv_cache[:, : w]))
        if ours.compressor is not None:
            pairs += [(ours.compressor.entries, a.kv_cache[:, w :]),
                      (ours.compressor.kv, a.compressor.kv_state),
                      (ours.compressor.score, a.compressor.score_state)]
        if ours.indexer is not None:
            pairs += [(ours.indexer.entries, a.indexer.kv_cache),
                      (ours.indexer.kv, a.indexer.compressor.kv_state),
                      (ours.indexer.score, a.indexer.compressor.score_state)]
    same_state = all(torch.equal(x, y) for x, y in pairs)
    print(f"s = {s:2d}  pos {state.pos:2d}  logits identical {same_logits}  state identical {same_state}  ({len(pairs)} tensors)")


In [ ]:
import sys
sys.path.insert(0, "checks")
from load_official import load_official, official_args

def memory_pairs(state, theirs, w):
    pairs = []
    for ours, layer in zip(state.layers, theirs.layers):
        a = layer.attn
        pairs.append((ours.window, a.kv_cache[:, : w]))
        if ours.compressor is not None:
            pairs += [(ours.compressor.entries, a.kv_cache[:, w :]),
                      (ours.compressor.kv, a.compressor.kv_state),
                      (ours.compressor.score, a.compressor.score_state)]
        if ours.indexer is not None:
            pairs += [(ours.indexer.entries, a.indexer.kv_cache),
                      (ours.indexer.kv, a.indexer.compressor.kv_state),
                      (ours.indexer.score, a.indexer.compressor.score_state)]
    return pairs

off = load_official("april")
args = tiny_args()
b, D = 2, 40

print("A. decode vs official, token by token (float32)")
for P in (1, 3, 4, 5, 8, 9, 16, 17, 33):
    torch.manual_seed(P)
    model = Transformer(args)
    theirs = off.Transformer(official_args(off, args, max_batch_size = b)).float()
    theirs.load_state_dict(model.state_dict(), strict = True)
    ids = torch.randint(0, args.vocab_size, (b, P + D))

    state = SequenceState(args, b)
    same = [torch.equal(model(ids[:, : P], state), theirs(ids[:, : P], 0))]
    for t in range(P, P + D):
        same.append(torch.equal(model(ids[:, t : t + 1], state), theirs(ids[:, t : t + 1], t)))
    same_memory = all(torch.equal(x, y) for x, y in memory_pairs(state, theirs, args.window_size))
    print(f"  P = {P:2d}  pos {state.pos}  logits identical at all {len(same)} steps: {all(same)}  memory identical: {same_memory}")

print("B. prefill + decode vs one prefill of everything (float64)")
args_b = tiny_args()
args_b.index_topk = 64 # the indexer keeps every entry, so tied scores can't change which ones it picks
for P in (1, 4, 5, 9, 17):
    torch.manual_seed(100 + P)
    model = Transformer(args_b).double()
    ids = torch.randint(0, args_b.vocab_size, (b, P + D))
    full = model(ids, return_all_logits = True) # [b, P + D, vocab]

    state = SequenceState(args_b, b, dtype = torch.float64)
    steps = [model(ids[:, : P], state)] # [b, vocab]
    for t in range(P, P + D):
        steps.append(model(ids[:, t : t + 1], state))
    steps = torch.stack(steps, dim = 1) # [b, D + 1, vocab]
    rel = ((steps - full[:, P - 1 :]).abs().max() / full.abs().max()).item()
    print(f"  P = {P:2d}  D = {D}  max relative difference {rel:.1e}")


In [ ]:
import time
torch.manual_seed(0)
args = tiny_args()
model = Transformer(args)
prompt = torch.randint(0, args.vocab_size, (2, 20)) # [b, P]

t0 = time.perf_counter()
with_memory = generate(model, args, prompt, 60) # [b, 60]
t1 = time.perf_counter()

ids = prompt
for _ in range(60):
    next_id = model(ids).argmax(dim = -1, keepdim = True) # [b, 1]: re-read everything every time
    ids = torch.cat([ids, next_id], dim = 1)
t2 = time.perf_counter()
without_memory = ids[:, 20 :] # [b, 60]

print(with_memory[0, : 12].tolist())
print("same tokens:", torch.equal(with_memory, without_memory))
print(f"with memory {t1 - t0:.2f} s, re-reading everything {t2 - t1:.2f} s")

In [ ]:
def memory(state):
    out = []
    for layer in state.layers:
        out.append(layer.window)
        for c in (layer.compressor, layer.indexer):
            if c is not None:
                out += [c.entries, c.kv, c.score]
    return out

args_c = tiny_args()
args_c.index_n_heads = 64 # like the real model; with 4 heads relu often zeroes all of them and scores tie at exactly 0
b, L = 2, 64

print("A. prefill P, then extends of T tokens, vs one prefill of all 64 tokens (float64)")
torch.manual_seed(0)
model = Transformer(args_c).double()
ids = torch.randint(0, args_c.vocab_size, (b, L))
full = model(ids, return_all_logits = True) # [b, L, vocab]
for P in (1, 3, 4, 5, 8, 9, 16, 17, 33):
    worst = 0.0
    for T in (1, 2, 3, 4, 5, 7, 8, 9, 16, 17):
        state = SequenceState(args_c, b, dtype = torch.float64)
        chunks = [model(ids[:, : P], state, return_all_logits = True)] # [b, P, vocab]
        while state.pos < L:
            chunks.append(model(ids[:, state.pos : state.pos + T], state, return_all_logits = True)) # [b, <= T, vocab]
        rel = ((torch.cat(chunks, dim = 1) - full).abs().max() / full.abs().max()).item()
        worst = max(worst, rel)
    print(f"  P = {P:2d}  T = 1..17  worst relative difference {worst:.1e}")

print("B. memory after one extend of T tokens vs after the same T tokens one at a time (float64)")
torch.manual_seed(1)
model = Transformer(args_c).double()
same = True
for P in (1, 3, 4, 5, 8, 9, 16, 17, 33):
    for T in (2, 3, 4, 5, 7, 8, 9, 16, 17):
        ids = torch.randint(0, args_c.vocab_size, (b, P + T))
        one_go = SequenceState(args_c, b, dtype = torch.float64)
        one_by_one = SequenceState(args_c, b, dtype = torch.float64)
        model(ids[:, : P], one_go)
        model(ids[:, : P], one_by_one)
        model(ids[:, P :], one_go)
        for t in range(P, P + T):
            model(ids[:, t : t + 1], one_by_one)
        same &= all(torch.allclose(x, y, rtol = 1e-12, atol = 1e-12) for x, y in zip(memory(one_go), memory(one_by_one)))
print(f"  81 cases, memory equal in all: {same}")


In [ ]:
def same_memory(s1, s2):
    pairs = []
    for l1, l2 in zip(s1.layers, s2.layers):
        pairs.append((l1.window, l2.window))
        for c1, c2 in ((l1.compressor, l2.compressor), (l1.indexer, l2.indexer)):
            if c1 is not None:
                pairs += [(c1.entries, c2.entries), (c1.kv, c2.kv), (c1.score, c2.score)]
    return s1.pos == s2.pos and all(torch.equal(x, y) for x, y in pairs)

args = tiny_args()
b = 2
torch.manual_seed(0)
model = Transformer(args)

print("A. snapshot -> wrong tokens -> restore -> right tokens, vs right tokens directly (float32)")
for P in (3, 4, 9, 16, 17, 33):
    for k in (1, 13):
        right = torch.randint(0, args.vocab_size, (b, P + 12))
        wrong = torch.randint(0, args.vocab_size, (b, k))

        direct = SequenceState(args, b)
        model(right[:, : P], direct)
        direct_out = [model(right[:, P : P + 5], direct, return_all_logits = True)]
        for t in range(P + 5, P + 12):
            direct_out.append(model(right[:, t : t + 1], direct, return_all_logits = True))

        state = SequenceState(args, b)
        model(right[:, : P], state)
        snap = state.clone()
        model(wrong, state)
        changed = not same_memory(state, snap)
        state.restore(snap)
        back = same_memory(state, snap)
        out = [model(right[:, P : P + 5], state, return_all_logits = True)]
        for t in range(P + 5, P + 12):
            out.append(model(right[:, t : t + 1], state, return_all_logits = True))
        same = all(torch.equal(x, y) for x, y in zip(out, direct_out)) and same_memory(state, direct)
        print(f"  P = {P:2d}  {k:2d} wrong tokens: memory changed {changed}  back to snapshot {back}  then identical to direct run {same}")

print("B. fork: two continuations from one prefix")
for P in (5, 17):
    prefix = torch.randint(0, args.vocab_size, (b, P))
    parent = SequenceState(args, b)
    model(prefix, parent)
    before = parent.clone()
    results = []
    for _ in range(2):
        cont = torch.randint(0, args.vocab_size, (b, 9))
        fork = parent.clone()
        direct = SequenceState(args, b)
        model(prefix, direct)
        results.append(torch.equal(model(cont, fork, return_all_logits = True), model(cont, direct, return_all_logits = True)))
    print(f"  P = {P:2d}: fork 1 identical to direct run {results[0]}  fork 2 {results[1]}  parent unchanged {same_memory(parent, before)}")

print("C. restore on a state created inside inference mode (as generate does)")
with torch.inference_mode():
    state = SequenceState(args, b)
model(right[:, : 5], state)
snap = state.clone()
model(wrong, state)
state.restore(snap)
print(f"  back to snapshot: {same_memory(state, snap)}")

In [ ]:
import sys
sys.path.insert(0, "checks")
from load_official import load_official

off = load_official("april")

print("A. fp8_simulate / fp4_simulate vs the official kernels")
torch.manual_seed(0)
edge = torch.zeros(2, 448)
edge[0, : 6] = torch.tensor([448.0, 1.0625, 1.1875, 17.0, 19.0, 0.0029296875])
edge[1, : 8] = torch.tensor([6.0, 0.25, 0.75, 1.25, 1.75, 2.5, 3.5, 5.0])
for dtype in (torch.float32, torch.bfloat16):
    same8 = same4 = True
    for x in [edge] + [torch.randn(64, 448) * scale for scale in (1e-6, 1e-2, 1.0, 300.0, 1e6)] + [torch.zeros(2, 448)]:
        x = x.to(dtype)
        ref8, ref4 = x.clone(), x.clone()
        off.act_quant(ref8, 64, "ue8m0", torch.float32, True)
        off.fp4_act_quant(ref4, 32, True)
        same8 &= torch.equal(fp8_simulate(x), ref8)
        same4 &= torch.equal(fp4_simulate(x), ref4)
    print(f"  {str(dtype):15s} FP8 identical {same8}  FP4 identical {same4}")

print("B. one block, before and after")
x = torch.zeros(448)
x[: 6] = torch.tensor([0.3, -1.7, 2.2, 0.05, 10.0, -3.3])
print("  x  ", [round(v, 4) for v in x[: 6].tolist()])
print("  FP8", [round(v, 4) for v in fp8_simulate(x)[: 6].tolist()])
print("  FP4", [round(v, 4) for v in fp4_simulate(x)[: 6].tolist()])

In [ ]:
import collections
from torch.utils._python_dispatch import TorchDispatchMode

class HostReads(TorchDispatchMode):
    def __init__(self):
        super().__init__()
        self.ops, self.reads = 0, collections.Counter()

    def __torch_dispatch__(self, func, types, args = (), kwargs = None):
        self.ops += 1
        name = func.overloadpacket.__name__
        if name in ("_local_scalar_dense", "nonzero", "bincount", "masked_select", "unique", "_unique2") or func is torch.ops.aten.where.default:
            self.reads[name] += 1
        return func(*args, **(kwargs or {}))

tolist_calls = 0
original_tolist = torch.Tensor.tolist
def counted_tolist(self):
    global tolist_calls
    tolist_calls += 1
    return original_tolist(self)

print("A. host reads in one prefill and one decode step (tiny, b = 1)")
args = tiny_args()
torch.manual_seed(0)
model = Transformer(args)
ids = torch.randint(0, args.vocab_size, (1, 21))
state = SequenceState(args, 1)
torch.Tensor.tolist = counted_tolist
try:
    for label, chunk in (("prefill 20 tokens", ids[:, : 20]), ("decode 1 token   ", ids[:, 20 :])):
        tolist_calls = 0
        with HostReads() as counter:
            model(chunk, state)
        print(f"  {label}: {counter.ops} ATen ops, other host reads {dict(counter.reads)}, routing copies {tolist_calls}")
finally:
    torch.Tensor.tolist = original_tolist

print("B. the cached Hadamard matrix also works outside inference mode, with gradients")
w = torch.nn.Parameter(torch.randn(32, 32))
rotate_activation(torch.randn(4, 32) @ w).sum().backward()
print("  backward ran:", w.grad is not None)


In [ ]:
import sys
sys.path.insert(0, "checks")
from load_official import load_official

off = load_official("0731")

print("A. sample vs the official sample, same random numbers")
torch.manual_seed(0)
same = []
for T in (0.0, 0.6, 1.0, 1.5):
    for shape in ((3, 512), (2, 5, 512)):
        logits = torch.randn(shape) * 3
        torch.manual_seed(7)
        ours = sample(logits, T)
        torch.manual_seed(7)
        same.append(torch.equal(ours, off.sample(logits, T)))
print(f"  identical in all {len(same)} cases: {all(same)}")

print("B. how often each token wins, vs softmax(logits / T), 200,000 draws")
logits = torch.tensor([2.0, 1.0, 0.0, -1.0])
g = torch.Generator().manual_seed(0)
for T in (1.0, 0.5):
    wins = sample(logits.expand(200_000, 4), T, g)
    want = [round(p, 3) for p in torch.softmax(logits / T, dim = -1).tolist()]
    got = [round(f, 3) for f in (torch.bincount(wins, minlength = 4) / 200_000).tolist()]
    print(f"  T = {T}: softmax {want}  measured {got}")

print("C. generate: same seed -> same tokens, another seed -> different tokens")
torch.manual_seed(0)
args = tiny_args()
model = Transformer(args)
prompt = torch.randint(0, args.vocab_size, (2, 20))
a = generate(model, args, prompt, 30, temperature = 1.0, seed = 1)
b = generate(model, args, prompt, 30, temperature = 1.0, seed = 1)
c = generate(model, args, prompt, 30, temperature = 1.0, seed = 2)
print("  seed 1:", a[0, : 10].tolist())
print("  seed 2:", c[0, : 10].tolist())
print(f"  seed 1 twice identical {torch.equal(a, b)}  seed 2 different {not torch.equal(a, c)}")


In [ ]:
import sys, json
sys.path.insert(0, "checks")
from load_official import load_official, official_args

off = load_official("0731")

print("A. ModelArgs() vs the official 0731 config.json")
cfg = json.load(open("deepseek_official/0731/inference/config.json"))
default, loaded = ModelArgs(), ModelArgs(**cfg)
diff = {k: (getattr(default, k), getattr(loaded, k)) for k in cfg if getattr(default, k) != getattr(loaded, k)}
print(f"  all {len(cfg)} fields equal: {not diff}  {diff if diff else ''}")

print("B. the hidden states DSpark reads (layers 3, 4, 5, averaged over the 4 streams) vs official 0731")
args = tiny_args()
b = 2
for P in (1, 5, 17):
    torch.manual_seed(P)
    model = Transformer(args)
    theirs = off.Transformer(official_args(off, args, max_batch_size = b)).float()
    theirs.load_state_dict(model.state_dict(), strict = True)
    ids = torch.randint(0, args.vocab_size, (b, P + 4))
    state = SequenceState(args, b)
    logits, hidden = model(ids[:, : P], state, return_hidden = True)
    _, their_logits, their_hidden = theirs(ids[:, : P], 0)
    same = [torch.equal(logits, their_logits), torch.equal(hidden, their_hidden)]
    shapes = [tuple(hidden.shape)]
    for t in range(P, P + 4):
        logits, hidden = model(ids[:, t : t + 1], state, return_hidden = True)
        _, their_logits, their_hidden = theirs(ids[:, t : t + 1], t)
        same += [torch.equal(logits, their_logits), torch.equal(hidden, their_hidden)]
    shapes.append(tuple(hidden.shape))
    print(f"  P = {P:2d}: hidden prefill {shapes[0]}, decode {shapes[1]}  identical at all 5 steps: {all(same)}")
logits, hidden = model(ids, return_all_logits = True, return_hidden = True)
print("  logits unchanged by return_hidden:", torch.equal(logits, model(ids, return_all_logits = True)))

print("C. the two heads vs official, same weights")
torch.manual_seed(0)
markov = DSparkMarkovHead(args.vocab_size, args.dspark_markov_rank)
confidence = DSparkConfidenceHead(args.dim + args.dspark_markov_rank)
their_markov = off.DSparkMarkovHead(args.vocab_size, args.dspark_markov_rank).float()
their_confidence = off.DSparkConfidenceHead(args.dim + args.dspark_markov_rank).float()
their_markov.load_state_dict(markov.state_dict(), strict = True)
their_confidence.load_state_dict(confidence.state_dict(), strict = True)
prev = torch.randint(0, args.vocab_size, (b,))
bias, emb = markov(prev)
their_bias, their_emb = their_markov(prev)
print(f"  Markov head: bias {tuple(bias.shape)}, emb {tuple(emb.shape)}  identical {torch.equal(bias, their_bias) and torch.equal(emb, their_emb)}")
x, m = torch.randn(b, 5, args.dim), torch.randn(b, 5, args.dspark_markov_rank)
score = confidence(x, m)
print(f"  confidence head: {tuple(score.shape)}  identical {torch.equal(score, their_confidence(x, m))}")

In [ ]:
import sys
sys.path.insert(0, "checks")
from load_official import load_official, official_args

off = load_official("0731")
args = tiny_args(dspark = True)
b, w, k = 2, args.window_size, args.dspark_block_size

print("A. SequenceState: one window per drafter stage")
state = SequenceState(args, b)
print(f"  main layers {len(state.layers)}, drafter windows {len(state.mtp)} of shape {tuple(state.mtp[0].window.shape)}, tensors() {len(state.tensors())}")
print(f"  drafter off (tiny_args()): {len(SequenceState(tiny_args(), b).mtp)} windows, tensors() {len(SequenceState(tiny_args(), b).tensors())}")

print("B. DSparkAttention vs official 0731 (float32): P main positions (memory only), then 10 steps of 1 main position + a block of 5")
with torch.inference_mode():
    for P in (1, 3, 8, 9, 17):
        torch.manual_seed(P)
        ours = DSparkAttention(args.n_layers, args)
        theirs = off.DSparkAttention(args.n_layers, official_args(off, args, max_batch_size = b)).float()
        theirs.load_state_dict(ours.state_dict(), strict = True)
        memory = SequenceState(args, b).mtp[0]
        x, main_x = torch.randn(b, k, args.dim), torch.randn(b, P, args.dim)
        same = [torch.equal(ours(x, memory, 0, main_x), theirs(x, 0, main_x))]
        for t in range(P, P + 10):
            x, main_x = torch.randn(b, k, args.dim), torch.randn(b, 1, args.dim)
            same.append(torch.equal(ours(x, memory, t, main_x), theirs(x, t, main_x)))
        print(f"  P = {P:2d}: outputs identical at all 11 steps {all(same)}  memory identical {torch.equal(memory.window, theirs.kv_cache[:b, : w])}")

print("C. s main positions at once, then the block == the same positions one at a time (float64)")
torch.manual_seed(0)
attn = DSparkAttention(args.n_layers, args).double()
worst, same_memory = 0.0, True
with torch.inference_mode():
    for P in (1, 3, 8, 9):
        for s in (2, 3, 5, 8, 9, 16):
            main_x = torch.randn(b, P + s, args.dim, dtype = torch.float64)
            x = torch.randn(b, k, args.dim, dtype = torch.float64)
            one_go = SequenceState(args, b, dtype = torch.float64).mtp[0]
            one_by_one = SequenceState(args, b, dtype = torch.float64).mtp[0]
            for m in (one_go, one_by_one):
                attn(x, m, 0, main_x[:, : P])
            y = attn(x, one_go, P, main_x[:, P :])
            for t in range(P, P + s):
                y_ref = attn(x, one_by_one, t, main_x[:, t : t + 1])
            worst = max(worst, ((y - y_ref).abs().max() / y_ref.abs().max()).item())
            same_memory &= torch.allclose(one_go.window, one_by_one.window, rtol = 1e-12, atol = 1e-12)
print(f"  24 cases: worst relative difference {worst:.1e}  memory equal {same_memory}")

print("D. snapshot / restore also covers the drafter's windows")
torch.manual_seed(1)
attn = DSparkAttention(args.n_layers, args)
state = SequenceState(args, b)
with torch.inference_mode():
    attn(torch.randn(b, k, args.dim), state.mtp[2], 0, torch.randn(b, 6, args.dim))
snap = state.clone()
with torch.inference_mode():
    attn(torch.randn(b, k, args.dim), state.mtp[2], 6, torch.randn(b, 1, args.dim))
changed = not torch.equal(state.mtp[2].window, snap.mtp[2].window)
state.restore(snap)
print(f"  one step changed the window {changed}  restore brought it back {torch.equal(state.mtp[2].window, snap.mtp[2].window)}")

print("E. a Block whose attention is DSparkAttention (what Step 22's stages will be) vs official 0731")
class TestBlock(Block):
    attention_cls = DSparkAttention

class TheirTestBlock(off.Block):
    attention_cls = off.DSparkAttention

with torch.inference_mode():
    torch.manual_seed(2)
    ours = TestBlock(args.n_layers, args)
    theirs = TheirTestBlock(args.n_layers, official_args(off, args, max_batch_size = b)).float()
    theirs.load_state_dict(ours.state_dict(), strict = True)
    memory = SequenceState(args, b).mtp[0]
    ids = torch.randint(0, args.vocab_size, (b, k))
    main_x = torch.randn(b, 9, args.dim)
    x = torch.randn(b, k, args.hc_mult, args.dim)
    same = [torch.equal(ours.attn(x, memory, 0, main_x), theirs.attn(x, 0, main_x))]
    for t in range(9, 15):
        x, main_x = torch.randn(b, k, args.hc_mult, args.dim), torch.randn(b, 1, args.dim)
        same.append(torch.equal(ours(x, ids, memory, t, main_x), theirs(x, t, ids, main_x)))
print(f"  attention is DSparkAttention: {type(ours.attn).__name__}  outputs identical at all 7 steps {all(same)}")

In [ ]:
import sys, json
sys.path.insert(0, "checks")
from load_official import load_official, official_args

off = load_official("0731")
args = tiny_args(dspark = True)
b, w = 2, args.window_size
aliases = sorted(f"mtp.{i}.{m}.weight" for i in range(3) for m in ("embed", "head"))

print("A. tiny model + drafter vs official 0731 (float32): prefill P (memory only), then 8 decode steps, each followed by a draft")
for P in (1, 8, 17):
    for T in (0.0, 1.0):
        torch.manual_seed(P)
        model = Transformer(args)
        theirs = off.Transformer(official_args(off, args, max_batch_size = b, n_mtp_layers = args.n_mtp_layers)).float()
        missing, unexpected = theirs.load_state_dict(model.state_dict(), strict = False)
        names_ok = sorted(missing) == aliases and not unexpected
        for stage in theirs.mtp:
            stage.temperature = T
        ids = torch.randint(0, args.vocab_size, (b, P + 8))
        state = SequenceState(args, b)
        logits, hidden = model(ids[:, : P], state, return_hidden = True)
        _, their_logits, their_hidden = theirs(ids[:, : P], 0)
        same = [model.forward_spec(logits.argmax(-1), hidden, state, T) is None and theirs.forward_spec(logits.argmax(-1), their_hidden, 0) is None]
        for t in range(P, P + 8):
            logits, hidden = model(ids[:, t : t + 1], state, return_hidden = True)
            _, their_logits, their_hidden = theirs(ids[:, t : t + 1], t)
            torch.manual_seed(1000 + t)
            ours = model.forward_spec(logits.argmax(-1), hidden, state, T)
            torch.manual_seed(1000 + t)
            ref = theirs.forward_spec(logits.argmax(-1), their_hidden, t)
            same += [torch.equal(logits, their_logits), torch.equal(hidden, their_hidden)] + [torch.equal(x, y) for x, y in zip(ours, ref)]
        memory = all(torch.equal(s.window, stage.attn.kv_cache[:b, : w]) for s, stage in zip(state.mtp, theirs.mtp))
        print(f"  P = {P:2d}  T = {T}: names {names_ok}  all {len(same)} outputs identical {all(same)}  drafter memory identical {memory}")

print("B. an extend of s main tokens, then a draft == the same tokens one at a time (float64)")
args64 = tiny_args(dspark = True)
args64.index_n_heads = 64
torch.manual_seed(0)
model = Transformer(args64).double()
worst, same_ids = 0.0, True
for P in (1, 4, 9):
    for s in (2, 5, 9):
        ids = torch.randint(0, args64.vocab_size, (b, P + s))
        one_go, one_by_one = SequenceState(args64, b, dtype = torch.float64), SequenceState(args64, b, dtype = torch.float64)
        for st in (one_go, one_by_one):
            logits, hidden = model(ids[:, : P], st, return_hidden = True)
            model.forward_spec(logits.argmax(-1), hidden, st, 0.0)
        logits, hidden = model(ids[:, P :], one_go, return_hidden = True)
        a = model.forward_spec(logits.argmax(-1), hidden, one_go, 0.0)
        for t in range(P, P + s):
            logits, hidden = model(ids[:, t : t + 1], one_by_one, return_hidden = True)
            c = model.forward_spec(logits.argmax(-1), hidden, one_by_one, 0.0)
        same_ids &= torch.equal(a[0], c[0])
        worst = max(worst, max(((x - y).abs().max() / y.abs().max()).item() for x, y in zip(a[1:], c[1:])))
print(f"  9 cases: guesses identical {same_ids}  worst relative difference (logits, confidence) {worst:.1e}")

print("C. our names for the real config (built on the meta device, no memory) vs the 0731 checkpoint")
with torch.device("meta"):
    real = Transformer(ModelArgs())
ours = set(real.state_dict())
index = json.load(open("deepseek_official/0731/model.safetensors.index.json"))["weight_map"]
ckpt = {k for k in index if not k.endswith(".scale")}
print(f"  ours {len(ours):,}  checkpoint {len(ckpt):,} (without .scale)  identical {ours == ckpt}  drafter names {len([k for k in ours if k.startswith('mtp.')]):,}")
del real
precompute_theta_cis.cache_clear()

print("D. one draft, up close (tiny random model, so the guesses mean nothing yet)")
torch.manual_seed(0)
model = Transformer(args)
state = SequenceState(args, 1)
prompt = torch.randint(0, args.vocab_size, (1, 9))
logits, hidden = model(prompt, state, return_hidden = True)
moe_calls = []
hooks = [stage.ffn.register_forward_hook(lambda *_: moe_calls.append(1)) for stage in model.mtp]
result = model.forward_spec(logits.argmax(-1), hidden, state)
for hook in hooks:
    hook.remove()
print(f"  prefill 9 tokens: hidden {tuple(hidden.shape)}, forward_spec returns {result}, stage MoE calls {len(moe_calls)}, memory only {len(moe_calls) == 0}")
next_id = logits.argmax(-1, keepdim = True)
logits, hidden = model(next_id, state, return_hidden = True)
guess_ids, guess_logits, confidence = model.forward_spec(logits.argmax(-1), hidden, state, 0.0)
print(f"  decode at position 9: hidden {tuple(hidden.shape)} -> ids {guess_ids[0].tolist()}  logits {tuple(guess_logits.shape)}  confidence {[round(v, 2) for v in confidence[0].tolist()]}")
print(f"  the first id is the main model's own pick {guess_ids[0, 0].item() == logits.argmax(-1).item()}; the next 5 are guesses for positions 11 to 15")


In [ ]:
torch.manual_seed(0)
chat = ChatTokenizer("deepseek_official/0731/tokenizer.json")
args = tiny_args()
args.vocab_size = 129_280
model = Transformer(args)

prompts = [
    "What is 2+2?", 
    "Write a haiku about the sea.", 
    "Explain in two sentences why the sky is blue during the day and red at sunset."
]

ids = [chat.encode_chat([{"role": "user", "content": p}]) for p in prompts]
print("prompt lengths:", [len(x) for x in ids])
print("first prompt ids:", ids[0])

new = generate(model, args, ids, 12, temperature = 1.0, seed = 0, eos_id = chat.eos_id) # [3, 12]
for p, row in zip(prompts, new.tolist()):
    print(f"{p[:28]:28s} -> {row}")
    print(f"{'':28s}    {chat.decode_completion(row)!r}")

In [ ]:
import sys
sys.path.insert(0, "checks")
from load_official import load_official, official_args, official_generate

print("A. tokenizer + official chat encoding")
chat = ChatTokenizer("deepseek_official/0731/tokenizer.json")
same = []
for k in (1, 2, 3, 4):
    text = open(f"deepseek_official/0731/encoding/tests/test_output_{k}.txt").read()
    same.append(chat.tokenizer.decode(chat.tokenizer.encode(text).ids, skip_special_tokens = False) == text)
print(f"  the 4 official example conversations come back unchanged after encode -> decode: {same}")
messages = [{"role": "user", "content": "What is 2+2?"}]
ids = chat.encode_chat(messages)
print(f"  'What is 2+2?' -> {len(ids)} ids {ids}")
print(f"  last id: chat mode {ids[-1]} (</think>), thinking mode {chat.encode_chat(messages, 'thinking')[-1]} (<think>)")
reply = chat.tokenizer.encode("2 + 2 = 4.").ids
for row in (reply + [chat.eos_id, 77, 88, -1, -1], reply + [-1, -1]):
    text = chat.decode_completion(row)
    print(f"  decode_completion({row}) = {text!r}  parse: {parse_message_from_completion_text(text, 'chat')['content']!r}")

print("B. generate vs the official generate (greedy, tiny model), prompts of different lengths")
off = load_official("0731")
their_generate = official_generate(off)
args = tiny_args()
torch.manual_seed(1)
prompts = [torch.randint(0, args.vocab_size, (n,)).tolist() for n in (11, 12, 21)]

def run(batch, eos_id):
    torch.manual_seed(0)
    model = Transformer(args)
    theirs = off.Transformer(official_args(off, args, max_batch_size = len(batch))).float()
    theirs.load_state_dict(model.state_dict(), strict = True)
    theirs.temperature = 0
    ours = generate(model, args, batch, 20, eos_id = eos_id) # [b, 20]
    ref = their_generate(theirs, batch, 20, eos_id)
    cut = [[t for t in row if t >= 0] for row in ours.tolist()]
    cut = [(row[: row.index(eos_id)] if eos_id in row else row) + [eos_id] for row in cut]
    return ours, cut == ref, [len(r) for r in ref]

free, same, lengths = run(prompts, args.vocab_size)
print(f"  lengths 11, 12, 21, no stop        identical {same}  completion lengths {lengths}")
for label, batch, eos_id in (("row 0 stops at its 5th token", prompts, free[0, 4].item()),
                             ("one prompt stops at its 7th token", prompts[2 :], free[2, 6].item()),
                             ("253-token prompt hits max_seq_len", [prompts[0] * 23], args.vocab_size),
                             ("end-of-text inside a prompt", [prompts[0], prompts[2][: 15] + [free[0, 2].item()] + prompts[2][16 :]], free[0, 2].item())):
    _, same, lengths = run(batch, eos_id)
    print(f"  {label:33s} identical {same}  completion lengths {lengths}")


In [ ]:
import sys
sys.path.insert(0, "checks")
from gates import run_gates
results = run_gates(globals(), real_size = RUN_REAL_SIZE)